# Llama 2 모델을 활용한 fine-tuning 및 RAG
- LoRa(Low Data Regime)를 활용한 적은 데이터에 대한 튜닝
- QLoRa 4비트 양자화 기법을 사용한 GPU 없에 파인튜닝 진행

1. peft: 대규모 모델을 효율적으로 미세 조정할 수 있게 돕는 라이브러리로, LoRA 등의 기법을 사용해 메모리와 계산 자원을 절약할 수 있습니다.
2. trl: 강화 학습 기반으로 텍스트 모델을 훈련하거나, 지도 학습을 통해 미세 조정할 수 있도록 돕는 라이브러리입니다. **SFTTrainer**는 지도 학습을 위한 훈련 도구입니다.

In [1]:
# !pip install -q accelerate==0.21.0 scipy tensorboardX peft==0.4.0 bitsandbytes==0.40.2 transformers==4.31.0 trl==0.4.7 tensorboardX

In [2]:
# !pip install transformers

## 01. 필요라이브러리 설정.

In [3]:
import os
import torch 
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM, # 인과적 언어 추론을 위한 모델을 자동으로 불러오는 클래스
    AutoTokenizer, # 입력 문장을 토큰 단위로 자동으로 잘라주는 역할
    BitsAndBytesConfig, # 모델 구성
    HfArgumentParser, # 파라미터 파싱
    TrainingArguments, # 훈련설정
    Pipeline, # 파이프라인 설정
    logging, # 로깅을 위한 클래스
)

# 모델 튜닝을 위한 라이브러리
from peft import LoraConfig, PeftModel
from trl import SFTTrainer

/opt/homebrew/anaconda3/envs/llm/lib/python3.9/site-packages/bitsandbytes/cextension.py:34: UserWarning: The installed version of bitsandbytes was compiled without GPU support. 8-bit optimizers, 8-bit multiplication, and GPU quantization are unavailable.
  warn("The installed version of bitsandbytes was compiled without GPU support. "


'NoneType' object has no attribute 'cadam32bit_grad_fp32'


## 02. 최초 사용 모델 및 fine-tuning에 적용할 데이터셋 정의.

In [4]:
# Hugging Face 허브에서 훈련하고자 하는 모델 가져와서 이름 지정

# 해당 부분에서는 사용 모델은 Bllossom.llama 을 사용 (단 튜닝은 추가적으로 진행해볼 예정
# model_name = "Bllossom/llama-3-Korean-Bllossom-70B"
model_name = "NousResearch/Llama-2-7b-chat-hf"

# instruction 데이터 세트 설정 => 해당 부분은 hugging-face에 업로드 되어 있는 데이터 셋을 추가 학습
# 시킨다는 목적이 있으며 차후 다른 데이터 셋 추가 학습이 필요한 경우 다른 데이터셋으로 훈련 진행
dataset_name = "mlabonne/guanaco-llama2-1k"

# fine-tuning(미세 조정)을 거친 후의 모델에 부여될 새로운 이름을 지정하는 변수
new_model = "kyusan-llama-2-chat-hf-mlabonne"

## 03. 저차원 가중치 행렬을 추가해 학습을 조정하기 위한 LoRA 파라이터 설정.

In [5]:
# LoRa에서 사용하는 low-rank matrices 어텐션 차원을 정의. (여기서는 64)
# 값이 클수록 더 많은 수정이 이루어지며, 모델이 복잡해질 수 있음.
# 값이 높을수록 오버피팅의 가능성을 높이지만 값이 낮을 수록 정보가 손실되어 데이터의 복잡한 특성을 포착하지 못할 수 있다.
lora_r =64

# LoRA 적용 시 가중치에 곱해지는 스케일링 요소. 여기서는 16으로 설정
# LoRA가 적용될 때 원래 모델의 가중치에 얼마나 영향을 미칠지 결정. 높은 값은 가중치 조정의 강도를 증가시킴

lora_alpha = 16

# Dropout probability for LoRA layers # LoRa 층에 적용되는 드롭아웃 확률
# 일부 네트워크 연결을 무작위로 비활성화하여 모델의 강건함에 기여
lora_dropout = 0.1

## 04. QLoRA 기법을 적용하기 위한 8비트 양자화 라이브러리 설정.

- 이 설정은 4-bit 양자화를 적용하여 모델의 메모리 사용량을 줄이고, 효율성을 극대화하는 방법입니다.
- Float16 타입을 사용해 계산 효율을 높이고, NF4 양자화 방식을 통해 양자화로 인한 성능 손실을 줄이려는 설정입니다.
- Nested Quantization(이중 양자화)은 비활성화되어, 더 극단적인 압축은 적용되지 않도록 설정되었습니다.

In [1]:
# 4-bit precision 기반의 모델 로드
use_4bit = True

# 4비트 기반 모델에 대한 dtype 계산
bnb_4bit_compute_dtype = "float16"
# => 메모리 부족으로 인한 float32 변경
# bnb_4bit_compute_dtype = "float32"

# 양자화 유형 (fp4 or nf4)
bnb_4bit_quant_type = "nf4"

# 4비트 기 모델에 대해 중첩 양자화 활성화 (이중 양자화)
# => 메모리 부족으로 인한 변경 False -> True
use_nested_quant = True


## 05. TrainingArguments 파라미터 설정
- hugging-face에서 제공하는 라이브러리로 모델의 학습부터 평가까지 한번에 해결할 수 있는 API를 제공

---
1. num_train_epochs 파라미터를 통해 모델이 전체 데이터셋을 몇 번 반복하여 학습할 지를 지정할 수 있고, 2. per_device_train_batch_size 를 통해 각 GPU에서 한 번에 처리할 데이터의 양을 나타낼 수도 있다. 여기에서는 모두 1로 지정하여 한 번에 하나의 데이터만 처리하도록 하였다.
3. gradient_accumulation_steps 파라미터를 통해 기울기를 갱신하기 전에 몇 번의 기울기 업데이트를 축적할 지를 결정하는데, 여기에서는 1로 설정되어 있어 매 스텝마다 기울기를 갱신한다.
4. 마지막으로 gradient_checkpointing 파라미터를 통해 메모리 사용을 최적화할 수 있다. 대규모 모델을 사용할 때 특히 유용한 파라미터로서 필요할 때만 특정 계층의 기울기를 저장하고 나머지는 버려 메모리의 부담을 줄인다.
5. max_grad_norm 파라미터를 통해 모델이 데이터로부터 학습하는 속도를 조절할 것이다. 기울기가 과도하게 커져서 발생할 수 있는 gradient exploding 문제 등을 방지할 수 있도록 기울기의 최대 크기를 설정한다.
6. Adam 옵티마이저를 사용할 것인데 이 때 learning_rate 를 보수적으로 잡아 모델이 데이터로부터 학습하는 속도를 적절히 늦추도록 할 것이다.
7. 마지막으로 weight_decay 값을 적절히 주어 모델의 가중치가 너무 큰 값을 가지지 않도록 함으로서 오버피팅 현상을 해소할 수 있다.
8. 스케줄러는 Cosine Decay를 사용하여 안정적으로 끊김없이 Loss가 감소하도록 할 것이다.

In [7]:
# 모델이 예측한 결과와 체크포인트가 저장될 출력 디렉터리
output_dir = "./checkpoint/"

# 훈련 에포크 수 
num_train_epochs = 1

# fp16/bf16 학습 활성화 
# A100으로 bf16을 True/ False 로 설정 => Colab을 이용한 고용량 서버 사용 시 가능 (현재 사용 X)
fp16 = False
bf16 = False

# 훈련용 배치 크기
per_device_train_batch_size = 1

# 평가용 배치 크기
per_device_eval_batch_size = 1

# 그래디언트(기울기)를 누적할 업데이트 스탭 횟수 
gradient_accumulation_step = 1

# 그래디언트 체크포인트 활성화 
gradient_checkpointing = True

# 그래디언트 클리핑을 위한 최대 그래디언트 노름을 설정.
# 그래디언트 클리핑은 그래디언트의 크기를 제안하여 훈련 중 안정성을 높임.
# Maximum gradient normal (그래디언트 클리핑) 0.3 으로 설정
max_grad_norm = 0.3

# 초기 학습률 Adam optimizer
learning_rate = 2e-6

# bias(편향)/LayerNorm 가중치를 제외하고 모든 레이어에 적용할 Weight decay (모델 정규화 (l2)) 값.
weight_decay = 0.001

# 옵디마이저 설정
optim = "paged_adamw_32bit" # 필요시 bit 수를 8bit 로 조정하여 적은 리소스로 학습 가능

# 학습률 스케줄러의 유형 설정, 여기서는 코사인 스케줄러 사용
# 코사인 함수를 기반으로 학습률을 변화시키는 방식을 뜻하며 처음에는 빠르게 학습하고 나중에는 천천히 수렴할 수 있도록 학습.
# 최종 단계에서 학습률을 매우 작게 조정하여 모델이 세밀하게 수렵할 수 있도록 도와줌
lr_scheduler_type = "cosine"

# 훈련 스탭 수(num_train_epochs 재정의) => 음수값인 -1은 훈련 스탭수를 직접 지정하지 않고 
# 훈련 에포크수에 의해 훈련을 조정한다는 의미
max_steps = -1

# (0부터 learning rate까지) 학습 초기에 학습률을 점진적으로 증가시기는 linear warmup 스탭의 Ratio
warmup_ratio = 0.03

# 시퀸스를 동일한 길이의 배치로 그룹화, 메모리 절약 및 훈련 속도를 높임
group_by_length = True

# N 업데이트 단계마다 체크포인트 저장
save_steps = 0

# 매 N 업데이트 스탭 로그
logging_steps = 25

## 06. SFT(Supervised Fine-Tuning) 파라미터 설정
- SFTTrainer로 import 한 라이브러리 사용
---
1. max_seq_length 는 입력 시퀀스에 대한 최대 사이즈를 의미한다. 예를 들어 문장 2개가 합쳐질 때 maximum sequence가 어느 정도일 지를 결정한다.
2. packing 이란 훈련 과정에서의 효율성을 높이기 위해 복수 개의 예시 문장을 하나의 Input 시퀀스로 넣어주는 기법을 의미한다.
3. device_map 을 통해서 몇 번 GPU를 로드할 지 지정할 수 있다.
(본인은 학습 환경이 GPU 1대였기 때문에 0번 GPU를 로드하도록 설정)


In [8]:
# 최대 시퀸스 길이 설정
max_seq_length = None

# 동일한 입력 시퀸스에 여러 개의 짧은 예제를 넣어 효율 성을 높일 수 있음.
packing = False

# GPU 0 전체 모델 로드
# device_map = {"": 0}
# device = torch.device("mps")
device = torch.device("cpu")
device_map = {"": device}

# device_map = {"": "cpu"}

In [9]:
device_map

{'': device(type='cpu')}

## 07. 데이터 셋 로딩과 데이터 타입 결정

In [10]:
dataset = load_dataset(dataset_name, split = "train")

In [11]:
# torch 라이브러리에서 bnb_4bit_compute_dtype 변수에 해당하는 데이터 타입을 가져오는 작업
# compute_dtype = getattr(torch, bnb_4bit_compute_dtype)
if torch.backends.mps.is_available():
    compute_dtype = torch.float32  # MPS 환경에 적합한 데이터 타입
else:
    compute_dtype = getattr(torch, bnb_4bit_compute_dtype)  # 기존 방식대로 가져옴


# 모델 계산에 사용될 데이터 타입 결정
bnb_config = BitsAndBytesConfig(
    load_in_4bit=use_4bit, # 모델을 4비트로 로드할지 여부 결정
    bnb_4bit_quant_type=bnb_4bit_quant_type, # 양자화 유형을 설정
    bnb_4bit_compute_dtype=compute_dtype, # 계산에 사용될 데이터 타입을 설정
    bnb_4bit_use_double_quant=use_nested_quant # 중첩 양자화를 사용할지 여부를 결정
)

## 08. GPU 호환성 확인 (window에서 확인 가능)
- MPS(Metal Performance Shaders)환경에서 작용할 수 있도록 적용 되는 코드

In [12]:
# GPU가 최소한 version 8 이상이라면 (major >= 8) bloat16을 지원한다고 메세지를 출력.
# bfloat16 은 훈련속도를 높일 수 있는 데이터 타입.

# if compute_dtype == torch.float16 and use_4bit:
#     major, _ = torch.cuda.get_device_capability()
#     if major >= 8:
#         print("=" * 80)
#         print("Your GPU supports bfloat16: accelerate training with bf16=True")
#         print("=" * 80)


##### CPU or Mac M1/M2
# device = torch.device("mps")
# device = torch.device("cpu")
# 모델을 적절한 장치(MPS 또는 CPU)로 이동 (필요시)
# model.to(device)

# 메모리 부족으로 인해 해당 부분 사용하지 않고 CPU를 기본적으로 사용
# if torch.backends.mps.is_available():
#     device = torch.device("mps")
# else:
#     device = torch.device("cpu")

## 09. 베이스 모델 로딩
- 훈련된 base 모델과 토크나이저를 로드한 다음 LoRa 연산을 적용

In [13]:
# Load base model
model = AutoModelForCausalLM.from_pretrained(
    model_name, 
    quantization_config = bnb_config,
    # device_map= device_map,
    device_map= {'':device},
    low_cpu_mem_usage=True  # 메모리 사용량을 줄이도록 설정
)

model.config.use_cache = False
model.config.pretraning_tp = 1

# Load LLaMA tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code = True)

# 동일한 batch 내에서 입력 크기를 동일하게 사용하기 위해서는 사용하는 padding token을 end of sequence라고,
# 하는 Special Token으로 사용한다.
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right" # Fix weird overflow issue with fp16 traning.

# Load LoRA configuration
peft_config = LoraConfig(
    lora_alpha=lora_alpha,
    lora_dropout=lora_dropout,
    r=lora_r,
    bias = "none",
    # Causal Language Modeling(CLM) 은 순차적인 텍스트 예측을 수행하는 모델을 의미.
    task_type = "CAUSAL_LM" # 파인튜닝할 테스크를 Optional 로 지정할 수 있는데, 여기서는 CASUAL_LM을 지정
)

# Set training parameters
training_arguments = TrainingArguments(
    output_dir = output_dir,
    num_train_epochs = num_train_epochs,
    per_device_eval_batch_size=per_device_eval_batch_size,
    gradient_accumulation_steps=gradient_accumulation_step,
    optim = optim,
    save_steps=save_steps,
    logging_steps=logging_steps,
    learning_rate=learning_rate,
    fp16=fp16,
    bf16=bf16,
    max_grad_norm=max_grad_norm,
    max_steps=max_steps,
    warmup_ratio=warmup_ratio,
    group_by_length=group_by_length,
    lr_scheduler_type=lr_scheduler_type,
    report_to= "tensorboard"
)

# Set supervised fine-tuning parameters
trainer = SFTTrainer(
    model = model,
    train_dataset=dataset,
    peft_config=peft_config,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    tokenizer=tokenizer,
    args = training_arguments,
    packing=packing
)

/opt/homebrew/anaconda3/envs/llm/lib/python3.9/site-packages/huggingface_hub/file_download.py:1142: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

/opt/homebrew/anaconda3/envs/llm/lib/python3.9/site-packages/peft/utils/other.py:102: FutureWarning: prepare_model_for_int8_training is deprecated and will be removed in a future version. Use prepare_model_for_kbit_training instead.
  warnings.warn(
/opt/homebrew/anaconda3/envs/llm/lib/python3.9/site-packages/trl/trainer/sft_trainer.py:159: UserWarning: You didn't pass a `max_seq_length` argument to the SFTTrainer, this will default to 1024
  warnings.warn(


## 10. 모델 훈련과 훈련된 모델 저장
- 기존 traniner 객체는 이전에 정의된 여러 설정 (모델, 데이터세트, 훈련 파라미터 등) 을 포함.
- train 메소드를 통해 데이터 세트를 반복적으로 처리하면서 모델의 가중치를 업데이트 하는 모습을 확인.

In [15]:
trainer.train()

# 훈련이 완료된 모델을 new_model에 저장
trainer.model.save_pretrained(new_model)

TypeError: device() received an invalid combination of arguments - got (NoneType), but expected one of:
 * (torch.device device)
      didn't match because some of the arguments have invalid types: (!NoneType!)
 * (str type, int index = -1)


## 11. 모델 이름 출력, 기본 모델 재로딩 후 LoRA 가중치와의 통합
- 모델과 LoRa를 따로 불러와 매핑핮 않고 하나의 모델로 활용
- merge_and_unload: 병합 및 배포

In [ ]:
# base_model과 new_model에 저장된 LoRA 가중치를 통합하여 새로운 모델을 생성
base_model = AutoModelForCausalLm.from_pretrained(
    model_name,
    low_cpu_mem_usage=True, 
    return_dict = True, 
    torch_dtype = torch.float32
)

model = PeftModel.from_pretrained(base_model, new_model) # LoRA 가중치를 가져와 기존 모델에 병합

In [ ]:
model = model.merge_and_unload()